# 04 — Rapport par organe, étape par étape

Ce notebook **reproduit chaque section** de la page produite par
`report_builder.build_rapport_organe`, avec les **graphiques inline**, pour l'organe
`Colon-Rectum-Anus` (`APPAREIL DIGESTIF`) au niveau `AP-HP` (exemple du notebook).
La vue page complète (IFrame) est conservée à la fin.

> ⚠ Duplique volontairement la logique de `build_rapport_organe` (branche `entity == 'AP-HP'`).
> Re-synchroniser si `report_builder` évolue.

In [1]:
# Parametres papermill
ORGANE   = 'Colon-Rectum-Anus'
APPAREIL = 'APPAREIL DIGESTIF'
ENTITY   = 'AP-HP'   # ou un GHU ex: 'GHU Nord'

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from pathlib import Path
import pandas as pd
from IPython.display import HTML

DATA_DIR   = Path('../data')
OUTPUT_DIR = Path('../output')
OUTPUT_DIR.mkdir(exist_ok=True)

from report_builder import load_aphp, load_regional, load_survival
from chart_utils import (
    line_evolution, donut_market_share, regional_comparison,
    survival_by_stage, survival_evolution, delay_evolution,
    TREATMENT_COLS, GHU_LIST, REGIONAL_COLORS,
)

aphp = load_aphp(DATA_DIR)
reg  = load_regional(DATA_DIR)
surv = load_survival(DATA_DIR)

# Sous-ensembles (identiques a build_rapport_organe)
org_data = aphp[(aphp.appareil == APPAREIL) & (aphp.organe == ORGANE)]
ent_org  = org_data[org_data.entite == ENTITY]
aphp_org = org_data[org_data.entite == 'AP-HP']

years = sorted(ent_org.annee.unique())
last_year, prev_year = years[-1], years[-2]
lv = ent_org[ent_org.annee == last_year].iloc[0]
pv = ent_org[ent_org.annee == prev_year].iloc[0]
print(f'Organe={ORGANE} appareil={APPAREIL} entity={ENTITY} - annees {years} - derniere={last_year}')

Organe=Colon-Rectum-Anus appareil=APPAREIL DIGESTIF entity=AP-HP - annees [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)] - derniere=2023


## Section 1 — Indicateurs clés

5 à 6 cartes KPI (incl. délai médian si présent). Vue tableau équivalente :

In [2]:
mesures = [
    (f'Patients {ENTITY}',     'nb_patients'),
    ('Nouveaux patients',      'nb_nouveaux_patients'),
    ('Sejours chirurgie',      'nb_sejours_chirurgie'),
    ('Sejours chimiotherapie', 'nb_sejours_chimiotherapie'),
    ('Sejours radiotherapie',  'nb_sejours_radiotherapie'),
]
if pd.notna(lv.get('delai_global_median')):
    mesures.append(('Delai median (j)', 'delai_global_median'))
pd.DataFrame([
    {'Indicateur': lab, str(prev_year): int(pv[col]), str(last_year): int(lv[col]),
     'Var. %': round((lv[col]-pv[col])/pv[col]*100, 1) if pv[col] else None}
    for lab, col in mesures
])

,Indicateur,2022,2023,Var. %
0,Patients AP-HP,2122,2161,1.8
1,Nouveaux patients,1054,1071,1.6
2,Sejours chirurgie,830,851,2.5
3,Sejours chimiotherapie,834,855,2.5
4,Sejours radiotherapie,581,596,2.6
5,Delai median (j),25,25,0.0


## Section 2 — Évolution

Branche `entity == 'AP-HP'` : le contexte est **régional**.
`regional_comparison` + `donut_market_share` (régional, dernière année) + séjours par mode de PEC.
Si l'organe est absent du régional, on retombe sur l'appareil (organe=TOTAL).

In [3]:
reg_org = reg[(reg.appareil == APPAREIL) & (reg.organe == ORGANE)]
if reg_org.empty:
    reg_org = reg[(reg.appareil == APPAREIL) & (reg.organe == 'TOTAL')]
fig_reg = regional_comparison(reg_org, 'nb_patients',
                              f'Contexte regional - {ORGANE}',
                              color_map=REGIONAL_COLORS)
fig_reg.show()

In [4]:
reg_org_last = reg_org[reg_org.annee == last_year]
fig_reg_donut = donut_market_share(
    reg_org_last, 'entite', 'nb_patients',
    f'Parts de marche regional - {ORGANE} ({last_year})',
    entities=sorted(reg_org_last['entite'].unique()), color_map=REGIONAL_COLORS)
fig_reg_donut.show()

In [5]:
melted = ent_org.melt(id_vars=['annee'], value_vars=list(TREATMENT_COLS.keys()),
                      var_name='type_sejour', value_name='nb_sejours')
melted['label'] = melted['type_sejour'].map(TREATMENT_COLS)
fig_sejours = line_evolution(melted, 'annee', 'nb_sejours', 'label',
                             f'Sejours par mode de PEC - {ORGANE}',
                             entities=list(TREATMENT_COLS.values()))
fig_sejours.show()

## Section 3 — Survie par stade

`survival_by_stage` + `survival_evolution` (stade I-III). Si la survie existe au niveau
organe, on l'utilise ; sinon on retombe au niveau appareil (logique du builder).

In [6]:
surv_org = surv[(surv.organe == ORGANE) & (surv.appareil == APPAREIL)]
if not surv_org.empty:
    fig_surv = survival_by_stage(surv, ENTITY, APPAREIL, organe=ORGANE, year=last_year)
else:
    fig_surv = survival_by_stage(surv, ENTITY, APPAREIL, year=last_year)
fig_surv.show()

In [7]:
if not surv_org.empty:
    fig_surv_evo = survival_evolution(surv, ENTITY, APPAREIL, organe=ORGANE, stade='I-III')
else:
    fig_surv_evo = survival_evolution(surv, ENTITY, APPAREIL, stade='I-III')
fig_surv_evo.show()

## Section 4 — Délais de prise en charge

`delay_evolution` pour cet organe.

In [8]:
fig_delay = delay_evolution(aphp, ENTITY, APPAREIL, organe=ORGANE)
fig_delay.show()

## Page complète (référence)

In [9]:
from report_builder import build_rapport_organe

out = build_rapport_organe(ORGANE, APPAREIL, DATA_DIR, OUTPUT_DIR,
                           entity=ENTITY, aphp=aphp, reg=reg, surv=surv)
print(f'Rapport genere : {out}')

from IPython.display import IFrame
IFrame(os.path.relpath(out), width='100%', height=800)

Rapport genere : ../output/rapport_organe_colon_rectum_anus.html
